In [1]:
%reset -f

In [2]:
from pynq import PL
from pynq import (allocate, Overlay)
import numpy as np
from PIL import Image
from datetime import datetime

PL.reset()

In [3]:
ol = Overlay('resizer-zcu102.bit')

In [4]:
help(ol)

Help on Overlay in module pynq.overlay:

<pynq.overlay.Overlay object>
    Default documentation for overlay resizer-zcu102.bit. The following
    attributes are available on this overlay:
    
    IP Blocks
    ----------
    axi_intc_0           : pynq.overlay.DefaultIP
    axi_vdma_0           : pynq.lib.video.dma.AxiVDMA
    img2axis_0           : pynq.overlay.DefaultIP
    sensorSupervisor_0   : pynq.overlay.DefaultIP
    zynq_ultra_ps_e_0    : pynq.overlay.DefaultIP
    
    Hierarchies
    -----------
    None
    
    Interrupts
    ----------
    None
    
    GPIO Outputs
    ------------
    None
    
    Memories
    ------------
    PSDDR                : Memory



In [5]:
def dump_s2mm_status(vdma_mmio):
    # S2MM register offsets
    S2MM_VDMACR = 0x30
    S2MM_VDMASR = 0x34
    S2MM_VSIZE  = 0xA0
    S2MM_HSIZE  = 0xA4
    S2MM_STRIDE = 0xA8

    # Read relevant registers
    cr = vdma_mmio.read(S2MM_VDMACR)
    sr = vdma_mmio.read(S2MM_VDMASR)
    vsize = vdma_mmio.read(S2MM_VSIZE)
    hsize = vdma_mmio.read(S2MM_HSIZE)
    stride = vdma_mmio.read(S2MM_STRIDE)

    print("----- VDMA S2MM Status Dump -----")
    print(f"Control Reg     (0x30): 0x{cr:08X}")
    print(f"Status Reg      (0x34): 0x{sr:08X}")
    print(f"Vertical Size   (0xA0): {vsize}")
    print(f"Horizontal Size (0xA4): {hsize}")
    print(f"Stride          (0xA8): {stride}")

    # Decode common status bits (optional)
    status_bits = {
        0:  "HALTED",
        1:  "VDMA Internal Error",
        2:  "Slave Error",
        3:  "Decode Error",
        4:  "Start of Frame Early Error",
        5:  "End of Line Early Error",
        6:  "Start of Frame Late Error",
        10: "End of Line Late Error",
        12: "Frame Count Interrupt",
        13: "Delay Count Interrupt",
        14: "Error Interrupt",
        31: "DMA Internal Halted"
    }

    print("Status Flags:")
    for bit, description in status_bits.items():
        if sr & (1 << bit):
            print(f" - Bit {bit}: {description}")

    print("----------------------------------")


In [6]:
# help(img2axis.register_map)

In [7]:

def img_to_axis(ip,buffer, eos,frame_cnt):
    
# Configure registers:
    # Write physical address of buffer to data_port register
    ip.register_map.data_port = buffer.physical_address

    # Set end_of_stream 
    ip.register_map.end_of_stream = eos

    # Set frame_no to 88
    ip.register_map.frame_cnt = frame_cnt
    # Start the IP core by setting the ap_start bit in CTRL register
    ip.register_map.CTRL.AP_START=1
    
def image_to_RGB(image_fname):
    # === LOAD AND CONVERT IMAGE TO RGB ===
    img = Image.open(f"{image_fname}").convert('RGB') 
    img_np = np.array(img)  # Shape: (H, W, 3), dtype=uint8

    # === PACK RGB TO INT32 ===
    # Format: 0x00RRGGBB (most significant byte can be 0)
    r = img_np[:, :, 0].astype(np.uint32)
    g = img_np[:, :, 1].astype(np.uint32)
    b = img_np[:, :, 2].astype(np.uint32)
    rgb_packed = (b << 16) | (g << 8) | r  # Shape: (H, W)

    # Allocate contiguous buffer with dtype uint32
    buffer = allocate(shape=rgb_packed.shape, dtype=np.uint32)

    # Copy packed pixels into buffer
    np.copyto(buffer, rgb_packed)

    print(f"Packed buffer shape: {buffer.shape}, dtype: {buffer.dtype}")
    return buffer

def save_img(fname, tensor):
    img_tensor = np.squeeze(tensor,axis=2)  # Remove batch dim → [C, H, W]
    img = Image.fromarray(img_tensor.astype(np.uint8), mode='L')  # 'L' = 8-bit pixels, black and white
    save_path=f"{fname}.png"
    img.save(save_path)
    return save_path

def unpack_frame(packed):
    H, W_packed,ch = packed.shape  # (480, 160,4)
    W = W_packed * ch            # Unpacked width = 640
    output_tensor = np.reshape(frame, (H, W, 1))
    return output_tensor

def save_frame(sufix, frame):
    print(f"type(frame)={type(frame)},  \
          frame.shape={frame.shape},frame.dtype={frame.dtype}")
    unpacked_frame  = frame.view(dtype=np.uint8).reshape((480, 640, 1))
    print(f"type(unpacked_frame)={type(unpacked_frame)},\nunpacked_frame.shape={unpacked_frame.shape}, \
          \nunpacked_frame.dtype={unpacked_frame.dtype}")
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    #fname=f"{timestamp}-{sufix}"
    fname=f"image-output-{sufix}"
    save_img(fname=fname, tensor=unpacked_frame)

In [8]:
import socket
import numpy as np
import time  # for simulating delay between frames


class Connection:
    _instance = None

    @classmethod
    def get_socket(cls, DEST_IP='192.168.100.119', DEST_PORT=5005):
        if cls._instance is None:
            cls._instance = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            cls._instance.connect((DEST_IP, DEST_PORT))
        return cls._instance

    @classmethod
    def reset_socket(cls):
        if cls._instance:
            cls._instance.close()
            cls._instance = None
            
    
def send_frame(unpacked_frame):

    sock = Connection.get_socket()
    # Remove the singleton channel dimension (shape becomes 480x640)
    frame_bytes = unpacked_frame.squeeze(axis=2).tobytes()


    # Optionally, send the frame size first (so the receiver knows what to expect)
    frame_size = len(frame_bytes)
    sock.sendall(frame_size.to_bytes(4, byteorder='big'))  # Send 4-byte length

    # Send the frame
    sock.sendall(frame_bytes)

    # Close the connection
    sock.close()

    print("Frame sent successfully.")
        

In [149]:
from pynq import MMIO, allocate
import numpy as np
import time

# ----------------------------------------
# VIVADO configuration
# ----------------------------------------
VDMA_BASE_ADDR = 0xA001_0000  # Replace with actual base address
VDMA_RANGE     = 0x1_0000    # 64K
FRAME_BUFFERS  = 2  
# ----------------------------------------
# FRAME configuration
# ----------------------------------------
WIDTH          = 160
HEIGHT         = 480
BPP            = 4           # Bytes per pixel (32bpp)
STRIDE         = WIDTH * BPP # Bytes per line
FRAME_SIZE     = STRIDE * HEIGHT

# ----------------------------------------
# Allocate destination buffer (for S2MM)
# ----------------------------------------
frame_rcv1 = allocate(shape=(HEIGHT, WIDTH), dtype=np.uint32, cacheable=1)
frame_rcv2 = allocate(shape=(HEIGHT, WIDTH), dtype=np.uint32, cacheable=1)
# ----------------------------------------
# MMIO object for VDMA
# ----------------------------------------
vdma_mmio = MMIO(VDMA_BASE_ADDR, VDMA_RANGE)

# ----------------------------------------
# S2MM register offsets
# ----------------------------------------
S2MM_VDMACR        =   0x30
S2MM_VDMASR        =   0x34
S2MM_VDMA_IRQ_MASK   =   0x3C
S2MM_REG_INDEX        =   0x44
S2MM_VSIZE        =   0xA0
S2MM_HSIZE        =   0xA4
S2MM_STRIDE     =   0xA8
S2MM_SA1        =   0xAC
S2MM_SA2        =   0xB0
S2MM_SA3        =   0xB4
S2MM_SA4        =   0xB8
S2MM_SA5        =   0xBC
S2MM_SA6        =   0xC0
S2MM_SA7        =   0xC4
S2MM_SA8        =   0xC8
S2MM_SA9        =   0xCC
S2MM_SA10       =   0xD0
S2MM_SA11       =   0xD4
S2MM_SA12       =   0xD8
S2MM_SA13       =   0xDC
S2MM_SA14       =   0xE0
S2MM_SA15       =   0xE4
S2MM_SA16       =   0xE8


# ----------------------------------------
# Reset S2MM Channel
# ----------------------------------------
vdma_mmio.write(S2MM_VDMACR , 0x00000004)  # Reset
time.sleep(0.01)
#vdma_mmio.write(S2MM_VDMACR , 0x00000001)  # S2MM_CONTROL: Run/Stop = 1, circular mode = 0
vdma_mmio.write(S2MM_VDMACR , 0x00000003)  # S2MM_CONTROL: Run/Stop = 1, circular mode = 1



# ----------------------------------------
# Set frame buffer base addresses
# ----------------------------------------
vdma_mmio.write(S2MM_REG_INDEX , 0x0)
vdma_mmio.write(S2MM_SA1 , frame_rcv1.physical_address)
vdma_mmio.write(S2MM_SA2 , frame_rcv2.physical_address)

# ----------------------------------------
# Set stride (bytes per row)
# ----------------------------------------
vdma_mmio.write(S2MM_STRIDE, STRIDE)

# ----------------------------------------
# Set horizontal size (in bytes)
# ----------------------------------------
vdma_mmio.write(S2MM_HSIZE, STRIDE)

# ----------------------------------------
# Set vertical size (in lines) to trigger transfer
# ----------------------------------------
vdma_mmio.write(S2MM_VSIZE, FRAME_BUFFERS*HEIGHT)




In [150]:
print(f"VDMA halted:{vdma_mmio.read(S2MM_VDMASR)&0x3}")
dump_s2mm_status(vdma_mmio)

VDMA halted:0
----- VDMA S2MM Status Dump -----
Control Reg     (0x30): 0x00010003
Status Reg      (0x34): 0x00010000
Vertical Size   (0xA0): 960
Horizontal Size (0xA4): 640
Stride          (0xA8): 640
Status Flags:
----------------------------------


In [151]:
img_fname1='1920x1080-full-hd-nature-landscape.jpg'
img_fname2='PM5644-1920x1080.gif'

In [152]:
buff_1=image_to_RGB(img_fname1)
buff_2=image_to_RGB(img_fname2)

Packed buffer shape: (1080, 1920), dtype: uint32
Packed buffer shape: (1080, 1920), dtype: uint32


In [153]:
frame_rcv1.fill(0)
frame_rcv2.fill(0)
img_to_axis(ol.img2axis_0,buff_1,False,FRAME_BUFFERS+1)
#img_to_axis(ol.img2axis_0,buff_1,False,1)
#img_to_axis(ol.img2axis_0,buff_2,True,1)

In [154]:
print(f"VDMA halted:{vdma_mmio.read(S2MM_VDMASR)&0x3}")
dump_s2mm_status(vdma_mmio)

VDMA halted:0
----- VDMA S2MM Status Dump -----
Control Reg     (0x30): 0x00010003
Status Reg      (0x34): 0x00015090
Vertical Size   (0xA0): 960
Horizontal Size (0xA4): 640
Stride          (0xA8): 640
Status Flags:
 - Bit 4: Start of Frame Early Error
 - Bit 12: Frame Count Interrupt
 - Bit 14: Error Interrupt
----------------------------------


In [155]:
save_frame('recv1',frame_rcv1)
save_frame('recv2',frame_rcv2)

type(frame)=<class 'pynq.buffer.PynqBuffer'>,            frame.shape=(480, 160),frame.dtype=uint32
type(unpacked_frame)=<class 'pynq.buffer.PynqBuffer'>,
unpacked_frame.shape=(480, 640, 1),           
unpacked_frame.dtype=uint8
type(frame)=<class 'pynq.buffer.PynqBuffer'>,            frame.shape=(480, 160),frame.dtype=uint32
type(unpacked_frame)=<class 'pynq.buffer.PynqBuffer'>,
unpacked_frame.shape=(480, 640, 1),           
unpacked_frame.dtype=uint8


In [16]:
#send_frame(unpacked_frame)

# playground